# NB 37 -- Packet P-019: High-Confidence Consensus Subset
## Agent OS v3.0 -- Materia Arche

**Decision gate:** Can we identify a high-confidence subset of devices where all three models (ExtraTrees, GradientBoosting, RandomForest) agree on rankings?

**Background:** P-011 showed only ~1/3 of panel devices have full multi-model consensus. ET-RF tau=0.80 but ET-GB tau=0.59 -- GradientBoosting diverges most. This packet defines a "consensus zone" based on rank spread across models and measures whether prediction quality improves within that zone.

In [1]:
"""Cell 2: Load data, train 3 models, predict on 20% test set."""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from scipy.stats import kendalltau

# --- data ---
df = pd.read_csv("perovskite_stability_clean.csv").fillna(0)

FEATURES = [
    "Perovskite_band_gap", "Pb", "Sn", "I", "Br", "Cl",
    "MA", "FA", "Cs",
    "first_Prvskt_annealing_temperature", "first_Prvskt_thermal_annealing_time",
    "Perovskite_thickness", "Cell_area_measured",
    "JV_default_Voc", "JV_default_Jsc", "JV_default_FF",
]

X = df[FEATURES].values
y = np.log1p(df["Stability_PCE_T80"].values)

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index.values, test_size=0.20, random_state=42
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# --- models (P-011 configs) ---
et = ExtraTreesRegressor(
    n_estimators=700, max_features="sqrt", min_samples_split=3,
    min_samples_leaf=1, bootstrap=False, random_state=42
)
gb = GradientBoostingRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.05, random_state=42
)
rf = RandomForestRegressor(
    n_estimators=700, max_features="sqrt", random_state=42
)

for name, model in [("ET", et), ("GB", gb), ("RF", rf)]:
    model.fit(X_train, y_train)
    print(f"  {name} trained")

pred_et = et.predict(X_test)
pred_gb = gb.predict(X_test)
pred_rf = rf.predict(X_test)
print("Predictions done.")

Train: 1234, Test: 309


  ET trained


  GB trained


  RF trained
Predictions done.


In [2]:
"""Cell 3: Compute per-device ranks, mean rank, rank spread, consensus flag."""
from scipy.stats import rankdata

rank_et = rankdata(-pred_et, method="average")  # 1 = highest predicted stability
rank_gb = rankdata(-pred_gb, method="average")
rank_rf = rankdata(-pred_rf, method="average")

results = pd.DataFrame({
    "dataset_index": idx_test,
    "actual_log1p_T80": y_test,
    "pred_ET": pred_et,
    "pred_GB": pred_gb,
    "pred_RF": pred_rf,
    "rank_ET": rank_et,
    "rank_GB": rank_gb,
    "rank_RF": rank_rf,
})

results["mean_rank"] = results[["rank_ET", "rank_GB", "rank_RF"]].mean(axis=1)
results["rank_spread"] = results[["rank_ET", "rank_GB", "rank_RF"]].max(axis=1) - \
                          results[["rank_ET", "rank_GB", "rank_RF"]].min(axis=1)

SPREAD_THRESHOLD = 30
results["consensus"] = results["rank_spread"] <= SPREAD_THRESHOLD

n_test = len(results)
n_consensus = results["consensus"].sum()
print(f"Test set size: {n_test}")
print(f"Consensus subset (spread <= {SPREAD_THRESHOLD}): {n_consensus} ({100*n_consensus/n_test:.1f}%)")
print(f"Non-consensus: {n_test - n_consensus}")
print(f"\nRank spread stats:")
print(results["rank_spread"].describe())

Test set size: 309
Consensus subset (spread <= 30): 96 (31.1%)
Non-consensus: 213

Rank spread stats:
count    309.000000
mean      57.440129
std       43.545697
min        0.000000
25%       26.000000
50%       47.000000
75%       81.000000
max      234.000000
Name: rank_spread, dtype: float64


In [3]:
"""Cell 4: Tau-b analysis -- consensus subset vs full test set."""

def compute_tau(a, b):
    tau, p = kendalltau(a, b)
    return tau, p

# Full test set tau-b
print("=== FULL TEST SET ===")
pairs = [("ET", "GB"), ("ET", "RF"), ("GB", "RF")]
full_taus = {}
for m1, m2 in pairs:
    tau, p = compute_tau(results[f"rank_{m1}"], results[f"rank_{m2}"])
    full_taus[(m1, m2)] = tau
    print(f"  {m1} <-> {m2}: tau-b = {tau:.4f}  (p = {p:.2e})")

# Actual vs each model (full)
print("\nActual vs model (full test set):")
actual_rank = rankdata(-results["actual_log1p_T80"].values, method="average")
full_actual_taus = {}
for m in ["ET", "GB", "RF"]:
    tau, p = compute_tau(actual_rank, results[f"rank_{m}"].values)
    full_actual_taus[m] = tau
    print(f"  Actual <-> {m}: tau-b = {tau:.4f}  (p = {p:.2e})")

# Consensus subset
cons = results[results["consensus"]].copy()
print(f"\n=== CONSENSUS SUBSET (n={len(cons)}) ===")
cons_actual_rank = rankdata(-cons["actual_log1p_T80"].values, method="average")
cons_taus = {}
for m1, m2 in pairs:
    tau, p = compute_tau(cons[f"rank_{m1}"].values, cons[f"rank_{m2}"].values)
    cons_taus[(m1, m2)] = tau
    print(f"  {m1} <-> {m2}: tau-b = {tau:.4f}  (p = {p:.2e})")

print("\nActual vs model (consensus subset):")
cons_actual_taus = {}
for m in ["ET", "GB", "RF"]:
    tau, p = compute_tau(cons_actual_rank, cons[f"rank_{m}"].values)
    cons_actual_taus[m] = tau
    print(f"  Actual <-> {m}: tau-b = {tau:.4f}  (p = {p:.2e})")

# Delta
print("\n=== DELTA (consensus - full) ===")
for m in ["ET", "GB", "RF"]:
    delta = cons_actual_taus[m] - full_actual_taus[m]
    print(f"  Actual <-> {m}: {delta:+.4f}")

=== FULL TEST SET ===
  ET <-> GB: tau-b = 0.5442  (p = 3.56e-46)
  ET <-> RF: tau-b = 0.7419  (p = 2.93e-84)
  GB <-> RF: tau-b = 0.6414  (p = 1.86e-63)

Actual vs model (full test set):
  Actual <-> ET: tau-b = 0.2714  (p = 1.34e-12)
  Actual <-> GB: tau-b = 0.2346  (p = 8.80e-10)
  Actual <-> RF: tau-b = 0.2618  (p = 8.01e-12)

=== CONSENSUS SUBSET (n=96) ===
  ET <-> GB: tau-b = 0.8924  (p = 6.61e-38)
  ET <-> RF: tau-b = 0.9087  (p = 3.10e-39)
  GB <-> RF: tau-b = 0.8766  (p = 1.23e-36)

Actual vs model (consensus subset):
  Actual <-> ET: tau-b = 0.3492  (p = 5.06e-07)
  Actual <-> GB: tau-b = 0.3611  (p = 2.05e-07)
  Actual <-> RF: tau-b = 0.3290  (p = 2.21e-06)

=== DELTA (consensus - full) ===
  Actual <-> ET: +0.0778
  Actual <-> GB: +0.1265
  Actual <-> RF: +0.0672


In [4]:
"""Cell 5: Profile consensus vs non-consensus devices."""

cons_mask = results["consensus"].values
noncons_mask = ~cons_mask

# Actual T80 comparison
actual_T80 = np.expm1(results["actual_log1p_T80"].values)
print("=== Actual T80 (hours) ===")
print(f"  Consensus (n={cons_mask.sum()}):     mean={actual_T80[cons_mask].mean():.1f}, median={np.median(actual_T80[cons_mask]):.1f}")
print(f"  Non-consensus (n={noncons_mask.sum()}): mean={actual_T80[noncons_mask].mean():.1f}, median={np.median(actual_T80[noncons_mask]):.1f}")

# Feature distributions
test_df = df.iloc[idx_test].copy()
test_df["consensus"] = results["consensus"].values

print("\n=== Feature means: Consensus vs Non-Consensus ===")
feat_compare = test_df.groupby("consensus")[FEATURES].mean().T
feat_compare.columns = ["Non-Consensus", "Consensus"]
feat_compare["Delta"] = feat_compare["Consensus"] - feat_compare["Non-Consensus"]
feat_compare["Abs_Delta"] = feat_compare["Delta"].abs()
feat_compare = feat_compare.sort_values("Abs_Delta", ascending=False)
print(feat_compare[["Non-Consensus", "Consensus", "Delta"]].to_string())

# Composition families
print("\n=== Composition families ===")
for group_label, mask in [("Consensus", cons_mask), ("Non-Consensus", noncons_mask)]:
    subset = test_df[mask]
    pure_MA = ((subset["MA"] > 0) & (subset["FA"] == 0) & (subset["Cs"] == 0)).sum()
    pure_FA = ((subset["FA"] > 0) & (subset["MA"] == 0) & (subset["Cs"] == 0)).sum()
    mixed = ((subset["MA"] > 0) & (subset["FA"] > 0)).sum()
    has_Cs = (subset["Cs"] > 0).sum()
    has_Sn = (subset["Sn"] > 0).sum()
    print(f"  {group_label} (n={len(subset)}): pure_MA={pure_MA}, pure_FA={pure_FA}, mixed_MA_FA={mixed}, has_Cs={has_Cs}, has_Sn={has_Sn}")

=== Actual T80 (hours) ===
  Consensus (n=96):     mean=421.6, median=168.0
  Non-consensus (n=213): mean=325.5, median=150.0

=== Feature means: Consensus vs Non-Consensus ===


                                     Non-Consensus   Consensus      Delta
Perovskite_thickness                    114.150235  176.875000  62.724765
first_Prvskt_annealing_temperature       90.492958   84.375000  -6.117958
first_Prvskt_thermal_annealing_time      22.281690   27.458333   5.176643
I                                         3.462277    6.215000   2.752723
MA                                        0.751016    1.716906   0.965890
Pb                                        1.193709    2.075521   0.881812
Cell_area_measured                        0.334055    0.110791  -0.223264
JV_default_Jsc                           19.084080   18.906771  -0.177309
Perovskite_band_gap                       1.121371    1.014813  -0.106558
FA                                        0.332411    0.263094  -0.069317
Br                                        0.229930    0.180833  -0.049096
JV_default_Voc                            0.961878    0.927344  -0.034534
Sn                                    

In [5]:
"""Cell 6: Check panel devices 850, 1320, 119."""
PANEL_DEVICES = [850, 1320, 119]

print("=== Panel Device Status ===")
for dev_id in PANEL_DEVICES:
    if dev_id in idx_test:
        row = results[results["dataset_index"] == dev_id].iloc[0]
        status = "CONSENSUS" if row["consensus"] else "NON-CONSENSUS"
        print(f"  Device {dev_id}: IN TEST SET -> {status}")
        print(f"    Ranks: ET={row['rank_ET']:.0f}, GB={row['rank_GB']:.0f}, RF={row['rank_RF']:.0f}")
        print(f"    Rank spread: {row['rank_spread']:.0f}, Mean rank: {row['mean_rank']:.1f}")
        print(f"    Actual log1p(T80): {row['actual_log1p_T80']:.3f}")
    else:
        print(f"  Device {dev_id}: NOT in test set (in training set for seed=42)")

=== Panel Device Status ===
  Device 850: NOT in test set (in training set for seed=42)
  Device 1320: IN TEST SET -> NON-CONSENSUS
    Ranks: ET=20, GB=68, RF=69
    Rank spread: 49, Mean rank: 52.3
    Actual log1p(T80): 6.847
  Device 119: NOT in test set (in training set for seed=42)


In [6]:
"""Cell 7: Threshold sweep -- rank spread from 10 to 100."""

thresholds = [10, 20, 30, 50, 100]
print(f"{'Threshold':>10} {'N_cons':>8} {'%_test':>8} {'tau_ET':>8} {'tau_GB':>8} {'tau_RF':>8} {'mean_tau':>9}")
print("-" * 70)

sweep_results = []
for thr in thresholds:
    mask = results["rank_spread"] <= thr
    n = mask.sum()
    pct = 100 * n / len(results)
    
    if n < 10:
        print(f"{thr:>10} {n:>8} {pct:>7.1f}% {'--':>8} {'--':>8} {'--':>8} {'--':>9}")
        sweep_results.append((thr, n, pct, None, None, None, None))
        continue
    
    sub = results[mask]
    sub_actual_rank = rankdata(-sub["actual_log1p_T80"].values, method="average")
    taus = {}
    for m in ["ET", "GB", "RF"]:
        tau, _ = kendalltau(sub_actual_rank, sub[f"rank_{m}"].values)
        taus[m] = tau
    mean_tau = np.mean(list(taus.values()))
    
    print(f"{thr:>10} {n:>8} {pct:>7.1f}% {taus['ET']:>8.4f} {taus['GB']:>8.4f} {taus['RF']:>8.4f} {mean_tau:>9.4f}")
    sweep_results.append((thr, n, pct, taus["ET"], taus["GB"], taus["RF"], mean_tau))

# Identify sweet spot
valid = [(t, n, p, te, tg, tr, mt) for t, n, p, te, tg, tr, mt in sweep_results if mt is not None and p >= 30]
if valid:
    best = max(valid, key=lambda x: x[6])
    print(f"\nSweet spot: threshold={best[0]}, n={best[1]} ({best[2]:.1f}%), mean_tau={best[6]:.4f}")
else:
    print("\nNo threshold achieves >=30% coverage with valid tau.")

 Threshold   N_cons   %_test   tau_ET   tau_GB   tau_RF  mean_tau
----------------------------------------------------------------------
        10       32    10.4%   0.3290   0.3532   0.3370    0.3397
        20       57    18.4%   0.2961   0.3062   0.2924    0.2982
        30       96    31.1%   0.3492   0.3611   0.3290    0.3464
        50      165    53.4%   0.3419   0.3447   0.3241    0.3369
       100      265    85.8%   0.3044   0.2946   0.3246    0.3079

Sweet spot: threshold=30, n=96 (31.1%), mean_tau=0.3464


In [7]:
"""Cell 8: Save CSV output."""
out = results[["dataset_index", "actual_log1p_T80",
               "pred_ET", "pred_GB", "pred_RF",
               "rank_ET", "rank_GB", "rank_RF",
               "mean_rank", "rank_spread", "consensus"]].copy()
out.to_csv("Packet_P019_High_Confidence_Consensus.csv", index=False)
print(f"Saved Packet_P019_High_Confidence_Consensus.csv  ({len(out)} rows)")

Saved Packet_P019_High_Confidence_Consensus.csv  (309 rows)


In [8]:
"""Cell 9: Honest status footer."""

n_test = len(results)
n_cons = results["consensus"].sum()
pct_cons = 100 * n_cons / n_test

# Best consensus actual tau (mean across models)
cons_sub = results[results["consensus"]]
cons_actual_rank_final = rankdata(-cons_sub["actual_log1p_T80"].values, method="average")
cons_tau_mean = np.mean([
    kendalltau(cons_actual_rank_final, cons_sub[f"rank_{m}"].values)[0]
    for m in ["ET", "GB", "RF"]
])

print("=" * 60)
print("P-019 HONEST STATUS FOOTER")
print("=" * 60)
print(f"Consensus subset: {n_cons}/{n_test} devices ({pct_cons:.1f}%)")
print(f"Mean actual-vs-model tau-b in consensus zone: {cons_tau_mean:.4f}")
print()

if pct_cons >= 50 and cons_tau_mean >= 0.30:
    status = "CONFIRMED"
    detail = f"Consensus covers {pct_cons:.0f}% (>=50%) with mean tau-b {cons_tau_mean:.3f} (>=0.30)"
elif pct_cons >= 30 and cons_tau_mean >= 0.20:
    status = "PROMISING"
    detail = f"Consensus covers {pct_cons:.0f}% (>=30%) with mean tau-b {cons_tau_mean:.3f} (>=0.20)"
else:
    status = "NEGATIVE"
    if pct_cons < 30:
        detail = f"Consensus covers only {pct_cons:.0f}% (<30%)"
    else:
        detail = f"Mean tau-b {cons_tau_mean:.3f} too low (<0.20)"

print(f"Status: {status}")
print(f"Detail: {detail}")
print("=" * 60)

P-019 HONEST STATUS FOOTER
Consensus subset: 96/309 devices (31.1%)
Mean actual-vs-model tau-b in consensus zone: 0.3464

Status: PROMISING
Detail: Consensus covers 31% (>=30%) with mean tau-b 0.346 (>=0.20)
